# AffineTensor tests

Regression tests for `intervalnets.affine.AffineTensor`.

In [ ]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_src = _cwd / 'src'
if not _src.exists():
    _src = _cwd.parent / 'src'
sys.path.insert(0, str(_src.resolve()))

from math import inf

import pytest

from intervalnets.affine import AffineTensor


In [ ]:
# test_point_has_zero_generators_and_exact_bounds
value = [1.0, -2.5]
affine = AffineTensor.point(value)
lower, upper = affine.to_bounds()
assert affine.c == (1.0, -2.5)
assert affine.G == ((), ())
assert lower[0] <= 1.0 <= upper[0]
assert lower[1] <= -2.5 <= upper[1]

In [ ]:
# test_from_bounds_round_trips_interval_conservatively
affine = AffineTensor.from_bounds([-1.0, 2.0], [3.0, 5.0])
lower, upper = affine.to_bounds()
assert lower[0] <= -1.0
assert upper[0] >= 3.0
assert lower[1] <= 2.0
assert upper[1] >= 5.0

In [ ]:
# test_affine_add_sub_and_negation_preserve_enclosure
a = AffineTensor.from_bounds([0.0, -1.0], [1.0, 2.0])
b = AffineTensor.from_bounds([-2.0, 1.0], [0.5, 3.0])
c = a + b
d = a - b
e = -a
c_lower, c_upper = c.to_bounds()
d_lower, d_upper = d.to_bounds()
e_lower, e_upper = e.to_bounds()
assert c_lower[0] <= -2.0 and c_upper[0] >= 1.5
assert c_lower[1] <= 0.0 and c_upper[1] >= 5.0
assert d_lower[0] <= -0.5 and d_upper[0] >= 3.0
assert d_lower[1] <= -4.0 and d_upper[1] >= 1.0
assert e_lower[0] <= -1.0 and e_upper[0] >= 0.0
assert e_lower[1] <= -2.0 and e_upper[1] >= 1.0

In [ ]:
# test_affine_scalar_arithmetic
a = AffineTensor.from_bounds([1.0, 2.0], [2.0, 4.0])
plus = a + 3.0
minus = 10.0 - a
plus_lower, plus_upper = plus.to_bounds()
minus_lower, minus_upper = minus.to_bounds()
assert plus_lower[0] <= 4.0 and plus_upper[0] >= 5.0
assert plus_lower[1] <= 5.0 and plus_upper[1] >= 7.0
assert minus_lower[0] <= 8.0 and minus_upper[0] >= 9.0
assert minus_lower[1] <= 6.0 and minus_upper[1] >= 8.0

In [ ]:
# test_affine_map_matches_matrix_rule_on_center_and_generators
z = AffineTensor.from_bounds([-1.0, 0.0], [3.0, 2.0])
W = ((2.0, -1.0), (0.5, 3.0))
b = (0.25, -2.0)
mapped = z.affine_map(W, b)
expected_center = (2.0 * z.c[0] - 1.0 * z.c[1] + 0.25, 0.5 * z.c[0] + 3.0 * z.c[1] - 2.0)
assert mapped.c == pytest.approx(expected_center)
lower, upper = mapped.to_bounds()
corners = [
    (2.0 * x - 1.0 * y + 0.25, 0.5 * x + 3.0 * y - 2.0)
    for x in (-1.0, 3.0)
    for y in (0.0, 2.0)
]
assert lower[0] <= min(c[0] for c in corners)
assert upper[0] >= max(c[0] for c in corners)
assert lower[1] <= min(c[1] for c in corners)
assert upper[1] >= max(c[1] for c in corners)

In [ ]:
# test_affine_map_dimension_validation
z = AffineTensor.from_bounds([-1.0, 0.0], [3.0, 2.0])
with pytest.raises(ValueError, match='Dimension mismatch'):
    _ = z.affine_map(((1.0, 2.0, 3.0),), (0.0,))

In [ ]:
# test_to_bounds_is_outward_rounded
z = AffineTensor.from_bounds(0.0, 1.0)
lower, upper = z.to_bounds()
assert lower < 0.0
assert upper > 1.0
assert lower != -inf and upper != inf

In [ ]:
# torch-backed affine activation transform setup
import torch

from intervalnets.affine_pytorch import (
    affine_relu_transform,
    affine_sigmoid_transform,
    affine_tanh_transform,
)


In [ ]:
# test_affine_activation_transforms_enclose_samples
def _assert_encloses_samples(transform_fn, point_fn, lower, upper, samples: int = 2000):
    x = AffineTensor.from_bounds(torch.tensor(lower, dtype=torch.float64), torch.tensor(upper, dtype=torch.float64))
    y = transform_fn(x)
    y_lower, y_upper = y.to_bounds()

    rand = torch.rand(samples, len(lower), dtype=torch.float64)
    lo = torch.tensor(lower, dtype=torch.float64)
    hi = torch.tensor(upper, dtype=torch.float64)
    xs = lo + (hi - lo) * rand
    ys = point_fn(xs)

    assert torch.all(ys >= y_lower.unsqueeze(0))
    assert torch.all(ys <= y_upper.unsqueeze(0))


_assert_encloses_samples(
    affine_relu_transform,
    lambda x: torch.relu(x),
    lower=[-2.0, -1.0, 0.2],
    upper=[3.0, 2.5, 1.4],
)

_assert_encloses_samples(
    affine_tanh_transform,
    lambda x: torch.tanh(x),
    lower=[-2.5, -0.5, 0.0],
    upper=[1.5, 2.0, 3.0],
)

_assert_encloses_samples(
    affine_sigmoid_transform,
    lambda x: torch.sigmoid(x),
    lower=[-6.0, -1.0, 0.2],
    upper=[-2.0, 3.0, 4.0],
)


In [ ]:
# test_affine_activation_transforms_handle_degenerate_intervals_exactly
point = torch.tensor([0.0, -1.5, 2.0], dtype=torch.float64)
x = AffineTensor.point(point)

relu_out = affine_relu_transform(x)
tanh_out = affine_tanh_transform(x)
sigmoid_out = affine_sigmoid_transform(x)

assert torch.allclose(relu_out.c, torch.relu(point))
assert torch.allclose(tanh_out.c, torch.tanh(point))
assert torch.allclose(sigmoid_out.c, torch.sigmoid(point))

zeros = torch.zeros(point.numel(), point.numel(), dtype=torch.float64)
assert torch.allclose(relu_out.G[:, -point.numel() :], zeros)
assert torch.allclose(tanh_out.G[:, -point.numel() :], zeros)
assert torch.allclose(sigmoid_out.G[:, -point.numel() :], zeros)
